[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Composition over Inheritance


## What you will be able to do

Tell whether a new class should inherit from another or hold one as a part, build classes out of
parts that each do one job, and explain why "prefer composition" is the usual advice.


## The idea

### The problem

The **Inheritance** notebook built `CoastalStation` on `Station`, and it fitted: a coastal station
is a station, and every method a station has makes sense on it. Most hierarchies are not built for
that reason. They are built to borrow.

Here is the usual case. A station's readings need somewhere to live that can be appended to,
looped over and measured with `len`, which is everything a `list` does. So
`class ListReadings(list):` looks like the obvious move, with `append` overridden to refuse `999`, a
temperature that does not occur on Earth. It passes the test that calls `append`.

But a list has other ways in. `extend` adds values without calling `append`, and so do `insert`,
`+=` and assignment to an index. Every one of them is inherited, none of them checks, and the class
cannot close them without overriding each one, including any that nobody thought of. It wanted one
method from `list` and got all of them, among them the ones that break its rule. That is what being
tied to everything the parent does looks like.

### What composition is

> **Composition** builds an object out of other objects. Instead of *being* a list, `Readings`
> *holds* one as an attribute. It offers only the methods it chooses to write, and each of them
> passes the work on to the object it holds, so nothing reaches the list except through those
> methods.

### Why it works that way

Inheritance decides a class's interface for you: whatever the parent can do, the child can do,
whether or not it makes sense. Composition leaves the interface to you. A `Readings` that holds a
list offers what it wrote, and nothing else exists to go around its check.

The test for which one you need has two questions. Is the new class genuinely a kind of the other,
so that `isinstance` ought to say yes? And would every method of the other class make sense on it?
A coastal station passes both. Readings fail the second: `sort` would destroy the order they were
taken in, and `clear` would erase the record.

Composition also keeps parts separate. A station that holds its readings, its site and its
formatting as three objects can swap any one of them while the program runs, and combining choices
costs one small class per choice rather than one class per combination. That last point is the
trouble with deep and multiple inheritance that the **Inheritance** notebook left for this one.

### Where you will meet this

The reading functions in the **Decorators** notebook used a `Connection` object handed to them,
rather than inheriting from one, and that is composition. It is also what makes a class easy to
test, because a test can hand it a fake part. The standard library leans the same way: `Path` holds
its text rather than being a `str`, which is why a path has `.suffix` and no `.upper()`.

### What this notebook covers

A list subclass against a class that holds a list, through the same numbered steps. Then the
two-question test, passing work to a part, swapping a part, the cost of a class per combination, and
where inheritance still belongs. Then one station assembled from parts.

### A first look

A class that holds a list instead of being one. There is nothing to run yet: read it, and read the
output underneath it.

```python
class Readings:
    def __init__(self):
        self._values = []

    def append(self, value):
        if not -90 <= value <= 60:
            raise ValueError(f"{value} is not a temperature on Earth")
        self._values.append(value)

    def __len__(self):
        return len(self._values)


readings = Readings()
readings.append(-4.1)
print(len(readings))
print(hasattr(readings, "extend"))
```

```
1
False
```

`Readings` offers only what it chose to write. There is no `extend` for anything to use around the
check.


## Setup

One import.

- `Path` is one of the standard library's own examples of holding rather than inheriting, used in one
  section

Every class in this notebook is written in the section that uses it.

**Run this cell before the rest of the notebook.**


In [1]:
from pathlib import Path

print("ready")


ready


## Worked examples

### Before and after: inheriting from `list`, or holding one

Here is the problem from the top of this notebook, in code. Both versions go through the same five
steps, numbered in the code and in the output:

1. Add two readings and ask for the mean.
2. Add an impossible reading, `999`, with `append`. It should be refused.
3. Add the same impossible reading with `extend`. It should be refused.
4. Replace the first reading with a word. It should be refused.
5. Ask for the mean again. It should still be `-3.35`.

First, the class that inherits from `list` and overrides `append` to check.


In [2]:
class ListReadings(list):
    """A station's readings, which must be temperatures that could really occur."""

    def append(self, value):
        if not -90 <= value <= 60:
            raise ValueError(f"{value} is not a temperature that occurs on Earth")
        super().append(value)

    def mean(self):
        return round(sum(self) / len(self), 2)


The five steps. Each step that might fail catches whichever error either version could raise, so the
lines can stay exactly the same for both.


In [3]:
readings = ListReadings()

# 1. Add two readings and ask for the mean.
readings.append(-4.1)
readings.append(-2.6)
print("1. mean:", readings.mean())

# 2. Add an impossible reading with append. It should be refused.
try:
    readings.append(999)
    print("2. accepted:", list(readings))
except ValueError as error:
    print("2. refused:", error)

# 3. Add an impossible reading with extend. It should be refused.
try:
    readings.extend([999])
    print("3. accepted:", list(readings))
except (ValueError, AttributeError) as error:
    print("3. refused:", error)

# 4. Replace the first reading with a word. It should be refused.
try:
    readings[0] = "warm"
    print("4. accepted:", list(readings))
except (ValueError, TypeError) as error:
    print("4. refused:", error)

# 5. Ask for the mean again. It should still be -3.35.
try:
    print("5. mean:", readings.mean())
except TypeError as error:
    print("5. crashed:", error)


1. mean: -3.35
2. refused: 999 is not a temperature that occurs on Earth
3. accepted: [-4.1, -2.6, 999]
4. accepted: ['warm', -2.6, 999]
5. crashed: unsupported operand type(s) for +: 'int' and 'str'


Step 2 was refused, because `append` is the one method `ListReadings` overrode. Step 3 used `extend`,
which `list` carries out without ever calling `append`, so the `999` went straight in. Step 4 replaced
a reading with a word through index assignment, another inherited way in. By step 5 the readings held
a word and an impossible number, and the mean crashed. The class checked one door out of many.

Here is every public method `ListReadings` inherited, and two operators that go around the check as
well.


In [4]:
print([name for name in dir(list) if not name.startswith("_")])

extra = ListReadings()
extra.append(-4.1)
extra += [999]
print("after += [999]:", extra, "and still a", type(extra).__name__)
print("extra + [999] is a", type(extra + [999]).__name__)


['append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']
after += [999]: [-4.1, 999] and still a ListReadings
extra + [999] is a list


Eleven public methods, and `append` is the only one that checks. `+=` got a `999` past it too, and
`+` did something stranger: it returned a plain `list`, so the result is no longer a `ListReadings`
and has no `mean`. To be safe, `ListReadings` would have to override every method that changes a
list, and the operators behind `+=` and indexing, and keep up with anything `list` gains later.

Now the same readings, held instead of inherited.


In [5]:
class Readings:
    """A station's readings, which must be temperatures that could really occur."""

    def __init__(self):
        self._values = []

    def append(self, value):
        if not -90 <= value <= 60:
            raise ValueError(f"{value} is not a temperature that occurs on Earth")
        self._values.append(value)

    def mean(self):
        return round(sum(self._values) / len(self._values), 2)

    def __len__(self):
        return len(self._values)

    def __iter__(self):
        return iter(self._values)


The same five steps. Only the class name on the first line is different.


In [6]:
readings = Readings()

# 1. Add two readings and ask for the mean.
readings.append(-4.1)
readings.append(-2.6)
print("1. mean:", readings.mean())

# 2. Add an impossible reading with append. It should be refused.
try:
    readings.append(999)
    print("2. accepted:", list(readings))
except ValueError as error:
    print("2. refused:", error)

# 3. Add an impossible reading with extend. It should be refused.
try:
    readings.extend([999])
    print("3. accepted:", list(readings))
except (ValueError, AttributeError) as error:
    print("3. refused:", error)

# 4. Replace the first reading with a word. It should be refused.
try:
    readings[0] = "warm"
    print("4. accepted:", list(readings))
except (ValueError, TypeError) as error:
    print("4. refused:", error)

# 5. Ask for the mean again. It should still be -3.35.
try:
    print("5. mean:", readings.mean())
except TypeError as error:
    print("5. crashed:", error)


1. mean: -3.35
2. refused: 999 is not a temperature that occurs on Earth
3. refused: 'Readings' object has no attribute 'extend'
4. refused: 'Readings' object does not support item assignment
5. mean: -3.35


Every mistake was refused. Step 3 failed because `Readings` has no `extend` at all, and step 4
because it does not support index assignment, so Python refused both before any value arrived. Step 5
gives `-3.35`, because nothing but the two good readings ever got in.

| | `ListReadings(list)` | `Readings`, holding a list |
|---|---|---|
| Ways to change the values | `append`, `extend`, `insert`, `+=`, index assignment and more | `append` only |
| Of those, checked | `append` | all of them, because `append` is the only one |
| Steps 3 and 4 | accepted `999` and `'warm'` | refused |
| Step 5 | crashed | `-3.35` |
| What callers can do | everything a `list` can | `append`, `mean`, `len` and a `for` loop |

The rest of this notebook takes the version that holds a list apart, and puts it to work.

| Question | The section that answers it |
|---|---|
| How do I tell whether to inherit or to hold? | Is it a kind of, or does it have one? |
| How does a station use the readings it holds? | Passing work to a part |
| What else does holding a part make possible? | Swapping a part |
| Why do deep hierarchies get out of hand? | One class per combination |
| When is inheriting still right? | When inheritance is right |

### Is it a kind of, or does it have one?

Two questions decide it. Is the new class genuinely a kind of the other one? And would every method
of the other one make sense on it?

| `list` method | On a station's readings |
|---|---|
| `append` | makes sense, with a check |
| `sort` | destroys the order the readings were taken in |
| `reverse` | the same |
| `clear` | erases the record |
| `insert` | puts a reading at a time it was not taken |
| `pop` | removes a reading without saying why |

Readings fail the second question badly, so they should hold a list, not be one. A coastal station
passes both questions, which is why the **Inheritance** notebook was right to subclass.

The standard library applies the same test.


In [7]:
path = Path("data/readings.csv")

print("Path is a kind of str:", issubclass(Path, str))
print("path.suffix:          ", path.suffix)
print("a path has .upper():  ", hasattr(path, "upper"))
print("str(path):            ", str(path))


Path is a kind of str: False
path.suffix:           .csv
a path has .upper():   False
str(path):             data/readings.csv


A path is made from text, and it is not a kind of text. `upper` would turn it into a path to a
different file, so `Path` does not offer it. It holds its text and offers path operations instead,
and `str(path)` hands the text back when you need it.

### Passing work to a part

A `Station` holds a `Readings` object and uses it. Each station method that concerns the readings
passes the work on to the part, which is called **delegation**.


In [8]:
class Station:
    def __init__(self, name):
        self.name = name
        self.readings = Readings()

    def record(self, value):
        self.readings.append(value)

    def report(self):
        return f"{self.name}: {len(self.readings)} readings, mean {self.readings.mean()}"


north = Station("Tromso")
north.record(-4.1)
north.record(-2.6)
print(north.report())

try:
    north.record(999)
except ValueError as error:
    print("refused through the station:", error)

print("isinstance(north, Readings):", isinstance(north, Readings))


Tromso: 2 readings, mean -3.35
refused through the station: 999 is not a temperature that occurs on Earth
isinstance(north, Readings): False


`Station.record` does not check anything. It passes the value to `Readings.append`, which checks, so
the rule lives in one place however many classes use readings. The station is not a kind of readings,
and `isinstance` agrees: it has some.

### Swapping a part

A part is an ordinary attribute, so it can be replaced while the program runs. Here a station holds a
formatter, an object whose only job is to turn a number into text.


In [9]:
class Celsius:
    def show(self, value):
        return f"{value:.1f} C"


class Fahrenheit:
    def show(self, value):
        return f"{value * 9 / 5 + 32:.1f} F"


class Station:
    def __init__(self, name, formatter):
        self.name = name
        self.readings = Readings()
        self.formatter = formatter

    def record(self, value):
        self.readings.append(value)

    def report(self):
        return f"{self.name}: mean {self.formatter.show(self.readings.mean())}"


north = Station("Tromso", Celsius())
north.record(-4.1)
north.record(-2.6)
print(north.report())

north.formatter = Fahrenheit()
print(north.report())


Tromso: mean -3.4 C
Tromso: mean 26.0 F


`north` is the same object before and after, and only one of its parts changed. With inheritance the
unit would be fixed by the class, `CelsiusStation` or `FahrenheitStation`, and changing it would mean
building a different object.

### One class per combination

Now suppose stations vary in two independent ways, where they are and which unit they report in. With
inheritance, every combination is a class: a coastal Celsius station, a coastal Fahrenheit station, and
so on. Each new choice multiplies the count. With composition, each option is one small part, and a
new choice only adds to it.


In [10]:
choices = [("location", 2), ("unit", 2), ("logging", 2), ("alert level", 3)]

classes_to_inherit, parts_to_compose = 1, 0
for choice, options in choices:
    classes_to_inherit *= options
    parts_to_compose += options
    print(f"after adding {choice:<12} {classes_to_inherit:>2} classes to inherit, "
          f"{parts_to_compose:>2} parts to compose")


after adding location      2 classes to inherit,  2 parts to compose
after adding unit          4 classes to inherit,  4 parts to compose
after adding logging       8 classes to inherit,  6 parts to compose
after adding alert level  24 classes to inherit,  9 parts to compose


Four choices make 24 classes one way and 9 parts the other, and the gap widens with every choice
added. This is the trouble with deep and multiple inheritance that the **Inheritance** notebook left for
this one: the hierarchy has to name every combination in advance.

With parts, every combination is built when it is needed.


In [11]:
class Coastal:
    def describe(self):
        return "coastal"


class Mountain:
    def describe(self):
        return "mountain"


class Station:
    def __init__(self, name, site, formatter):
        self.name = name
        self.site = site
        self.formatter = formatter
        self.readings = Readings()

    def report(self):
        mean = self.formatter.show(self.readings.mean())
        return f"{self.name} ({self.site.describe()}): mean {mean}"


for site in [Coastal(), Mountain()]:
    for formatter in [Celsius(), Fahrenheit()]:
        station = Station("Test", site, formatter)
        station.readings.append(-4.1)
        station.readings.append(-2.6)
        print(station.report())


Test (coastal): mean -3.4 C
Test (coastal): mean 26.0 F
Test (mountain): mean -3.4 C
Test (mountain): mean 26.0 F


Four combinations from four small parts and one `Station` class. A third choice would add its options
as parts, and `Station` would not need to change.

### When inheritance is right

Composition is the default, not the rule. When a class genuinely is a kind of another, and every
method of the other applies, inheriting is correct. Exceptions are the clearest case.


In [12]:
class ReadingError(ValueError):
    pass


try:
    raise ReadingError("999 is not a temperature that occurs on Earth")
except ValueError as error:
    print("except ValueError caught a", type(error).__name__)


except ValueError caught a ReadingError


A `ReadingError` is a kind of `ValueError`, and code that already catches `ValueError` should catch it
too. Holding a `ValueError` as a part would lose exactly that. The **Exceptions as Classes** notebook
builds on this.

| The new class, and the other | Inherit or hold? | Why |
|---|---|---|
| `CoastalStation` and `Station` | inherit | a kind of station, and every station method applies |
| `ReadingError` and `ValueError` | inherit | a kind of `ValueError`, and `except ValueError` should catch it |
| `Readings` and `list` | hold | readings have a list, and `sort` or `clear` would break them |
| `Station` and `Readings` | hold | a station has readings; it is not readings |
| `Station` and a formatter | hold | the formatter is a choice that can change |

Inherit when both answers are yes. Hold in every other case.

### Putting it together: a station assembled from parts

One station class and four small parts. The readings check every value, the site describes itself,
and the formatter turns numbers into text. The site takes the station's formatter as an argument, so
the sea temperature of a coastal site is shown in the same unit as everything else.

The parts default to `None` and are built inside `__init__`, for the reason the quiet error at the end
explains.


In [13]:
class Inland:
    def describe(self, formatter):
        return "inland"


class Coastal:
    def __init__(self, sea):
        self.sea = sea

    def describe(self, formatter):
        return f"coastal, sea {formatter.show(self.sea)}"


class Station:
    """A station assembled from parts: its readings, its site, and how it shows a value."""

    def __init__(self, name, site, formatter=None, readings=None):
        self.name = name
        self.site = site
        self.formatter = Celsius() if formatter is None else formatter
        self.readings = Readings() if readings is None else readings

    def record(self, value):
        self.readings.append(value)

    def report(self):
        mean = self.formatter.show(self.readings.mean())
        where = self.site.describe(self.formatter)
        return f"{self.name} ({where}): {len(self.readings)} readings, mean {mean}"


network = [
    Station("Tromso", Inland()),
    Station("Bergen", Coastal(sea=7.1)),
    Station("Malaga", Coastal(sea=18.2), Fahrenheit()),
]

for station, values in zip(network, [[-4.1, -2.6], [3.1, 4.4], [18.9, 19.4]]):
    for value in values:
        station.record(value)

for station in network:
    print(station.report())


Tromso (inland): 2 readings, mean -3.4 C
Bergen (coastal, sea 7.1 C): 2 readings, mean 3.8 C
Malaga (coastal, sea 64.8 F): 2 readings, mean 66.5 F


Three stations built from different parts, reported by one method. Malaga was given a Fahrenheit
formatter, and its sea temperature followed.

Now an impossible reading, and a change of unit for the whole network.


In [14]:
try:
    network[0].record(999)
except ValueError as error:
    print("refused:", error)

for station in network:
    station.formatter = Fahrenheit()

for station in network:
    print(station.report())


refused: 999 is not a temperature that occurs on Earth
Tromso (inland): 2 readings, mean 26.0 F
Bergen (coastal, sea 44.8 F): 2 readings, mean 38.8 F
Malaga (coastal, sea 64.8 F): 2 readings, mean 66.5 F


The `999` was refused by `Readings`, the one place the rule lives. Switching the network to Fahrenheit
replaced one part in each station, and the means and the sea temperatures both followed, because both
are shown through that part.

### Where each part came from

| In the network | What it relies on | The section that showed it |
|---|---|---|
| `Readings` refusing `999` from any station | a class that holds a list offers only what it wrote | Before and after |
| `record` and `report` using `self.readings` | passing work to a part | Passing work to a part |
| the whole network switched to Fahrenheit | a part is an attribute, and can be replaced | Swapping a part |
| inland and coastal sites in one class of station | one part per option, not one class per combination | One class per combination |
| parts that default to `None` | a default part must not be shared | the quiet error, below |
| `Station` holding, not inheriting, all four | a station has readings, a site and a format | Is it a kind of, or does it have one? |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/10-composition-over-inheritance-solutions.ipynb).

**1.** Write a `Log` class that holds a list of messages, with an `add(message)` method and a
`__len__`. Add two messages and print `len(log)`.


In [15]:
# your code here


**2.** Give `Log` an `__iter__`, so that a `for` loop can print every message.


In [16]:
# your code here


**3.** Try `log.clear()` and `log[0] = "x"` on your `Log`, catching each error and printing it. Say in
a comment why those errors are the point.


In [17]:
# your code here


**4.** Write a `Station` that holds a `Log`, with a `note(text)` method that passes the text to it.
Make two notes, then print them by looping over the station's log.


In [18]:
# your code here


**5.** Give `Station` a `formatter` part, using `Celsius` and `Fahrenheit` classes that each have a
`show(value)` method. Print one station's report, swap its formatter, and print the report again.


In [19]:
# your code here


**6.** No code for this one. For each pair below, write inherit or hold in a comment, and one sentence
of why.

- `UrgentAlert` and `Alert`, where every `Alert` method applies to an urgent alert.
- `Inventory` and `dict`, where `Inventory` wants to borrow `get`.
- `Timer` and `Station`, where `Timer` wants to reuse `report`.


In [20]:
# your code here


## Common errors

### TypeError: `len` on a class that holds a list

Holding a list does not make the class behave like one. `len` works only if the class says what its
length is.


In [21]:
class NoLength:
    def __init__(self):
        self._values = []

    def append(self, value):
        self._values.append(value)


readings = NoLength()
readings.append(-4.1)

len(readings)


TypeError: object of type 'NoLength' has no len()

`object of type 'NoLength' has no len()`. The list inside has a length, and `len` does not look inside.
A class that holds a collection writes the protocols it wants to support, `__len__` for `len` and
`__iter__` for `for`, each passing the work on, as `Readings` did. That is the cost of composition: a
few one-line methods, in return for choosing exactly what callers get.

### AttributeError: asking the whole for the part's method

With inheritance, a subclass has its parent's methods. With composition, the methods stay on the part.


In [22]:
class Station:
    def __init__(self, name):
        self.name = name
        self.readings = Readings()

    def record(self, value):
        self.readings.append(value)


north = Station("Tromso")
north.record(-4.1)

north.mean()


AttributeError: 'Station' object has no attribute 'mean'

`mean` belongs to the `Readings` that `north` holds, so the call is `north.readings.mean()`. If callers
should be able to ask the station directly, the station writes a `mean` of its own that passes the call
on, which is delegation again.

### AttributeError: a list subclass handed back a plain list

`+` on a list subclass runs the `list` version, which builds and returns a `list`.


In [23]:
recent = ListReadings()
recent.append(-4.1)

combined = recent + [-2.6]
print("combined is a", type(combined).__name__)

combined.mean()


combined is a list


AttributeError: 'list' object has no attribute 'mean'

The subclass was lost in the addition, and its `mean` with it. Nothing flagged the change of type at
the `+`; the error waited for the first method only `ListReadings` has. Every `list` operation that
builds a new list does the same, which is one more way a subclass of a built-in type stops being the
class you wrote.

### The quiet one: a default part made once

A part given as a default argument is built once, when the `def` runs, and shared by every object that
takes the default. It is the mutable default from the **Your First Class** notebook, arriving as a part.


In [24]:
class SharedDefault:
    def __init__(self, name, readings=Readings()):
        self.name = name
        self.readings = readings

    def record(self, value):
        self.readings.append(value)


tromso = SharedDefault("Tromso")
malaga = SharedDefault("Malaga")
tromso.record(-4.1)

print("Tromso's readings:", list(tromso.readings))
print("Malaga's readings:", list(malaga.readings))
print("one part:         ", tromso.readings is malaga.readings)


Tromso's readings: [-4.1]
Malaga's readings: [-4.1]
one part:          True


A reading recorded at Tromso appeared at Malaga, and nothing raised. `Readings()` in the `def` line ran
once, and every station made without its own readings was handed that one object.

Default to `None`, and build the part inside `__init__`, as the network's `Station` did.


In [25]:
class Separate:
    def __init__(self, name, readings=None):
        self.name = name
        self.readings = Readings() if readings is None else readings


tromso = Separate("Tromso")
malaga = Separate("Malaga")
tromso.readings.append(-4.1)

print("Tromso's readings:", list(tromso.readings))
print("Malaga's readings:", list(malaga.readings))
print("one part:         ", tromso.readings is malaga.readings)


Tromso's readings: [-4.1]
Malaga's readings: []
one part:          False


Each station now gets its own part, because `Readings()` runs on every call rather than once.


## Recap

- Inheriting to borrow one method gives a class every method of its parent, including the ones that
  break its rules.
- A list subclass that checks in `append` is still open through `extend`, `insert`, `+=` and index
  assignment.
- Composition holds a part as an attribute and offers only the methods it writes.
- Each of those methods passes the work on to the part, which is called delegation.
- Ask two questions: is it a kind of the other, and would every method of the other make sense on it?
  Inherit only when both answers are yes.
- A class that holds a list needs `__len__` and `__iter__` before `len` and `for` work on it.
- A part can be swapped while the program runs, which a class hierarchy cannot do.
- Combining choices costs one class per choice with composition, and one class per combination with
  inheritance.
- Operators on a list subclass can return a plain `list`, dropping the subclass on the way.
- A part given as a default argument is made once and shared. Default to `None` and build it inside.
- Hold by default. Inherit for genuine kinds, such as a new exception.


## What is next

The **Dataclasses** notebook. Every class in this guide has spent lines on the same chores: an
`__init__` that copies its arguments onto `self`, a `__repr__`, an `__eq__`. `@dataclass` writes all
three from a list of fields, and it is where the classes in this guide get shorter.


---

&#8592; **Previous:** [Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/09-inheritance.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
